# Book recommendation demo

Choose a raw Book-Crossing `User-ID` and either the implicit `mlp` or `neumf` checkpoint. The notebook shows:

1. the five highest implicit interaction probabilities;
2. the five highest explicit rating predictions;
3. a hybrid ranking that takes the explicit model's top 1,000 eligible books and sorts them by `explicit_score * implicit_probability`.

Books the user has already interacted with are excluded. The hybrid is a simple score-fusion heuristic, not a separately trained model.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from bookrec.data import ITEM_COLUMN, RATING_COLUMN, USER_COLUMN, load_dataset
from bookrec.explicit.hyperparameters import (
    DEFAULT_HYPERPARAMETERS as EXPLICIT_DEFAULTS,
)
from bookrec.explicit.model import ExplicitRecommenderMLP
from bookrec.implicit.hyperparameters import (
    DEFAULT_HYPERPARAMETERS as IMPLICIT_DEFAULTS,
)
from bookrec.implicit.model import MODEL_REGISTRY, create_implicit_model

ARTIFACT_ROOT = Path("artifacts")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INFERENCE_BATCH_SIZE = 16_384

print(f"Inference device: {DEVICE}")

In [ ]:
ratings = load_dataset("Ratings.csv")
books = load_dataset("Books.csv")
ratings[ITEM_COLUMN] = ratings[ITEM_COLUMN].astype(str)
books[ITEM_COLUMN] = books[ITEM_COLUMN].astype(str)

book_metadata = (
    books[[
        ITEM_COLUMN,
        "Book-Title",
        "Book-Author",
        "Year-Of-Publication",
        "Publisher",
        "Image-URL-M",
    ]]
    .drop_duplicates(subset=[ITEM_COLUMN])
    .reset_index(drop=True)
)
metadata_isbns = set(book_metadata[ITEM_COLUMN])

implicit_reference = torch.load(
    ARTIFACT_ROOT / "implicit" / "mlp" / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
explicit_reference = torch.load(
    ARTIFACT_ROOT / "explicit" / "mlp" / "model_with_mappings.pt",
    map_location="cpu",
    weights_only=False,
)
eligible_users = (
    set(implicit_reference["user_to_index"])
    & set(explicit_reference["user_to_index"])
)
user_activity = (
    ratings[ratings[USER_COLUMN].isin(eligible_users)]
    .groupby(USER_COLUMN)
    .agg(
        interactions=(ITEM_COLUMN, "size"),
        explicit_ratings=(RATING_COLUMN, lambda values: int((values > 0).sum())),
    )
    .query("explicit_ratings > 0")
    .sort_values(["explicit_ratings", "interactions"], ascending=False)
)

print(f"Users available to both models: {len(eligible_users):,}")
display(user_activity.head(20))

## Select the user and implicit model

Use a `User-ID` from the table above. Change `IMPLICIT_MODEL_NAME` to `"mlp"` or `"neumf"`, then rerun this cell and the final cell.

In [ ]:
USER_ID = 11676
IMPLICIT_MODEL_NAME = "mlp"  # choose "mlp" or "neumf"
TOP_K = 5
EXPLICIT_POOL_SIZE = 1_000

In [ ]:
def load_implicit_model(model_name: str):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"model_name must be one of {tuple(MODEL_REGISTRY)}")

    checkpoint_path = (
        ARTIFACT_ROOT
        / "implicit"
        / model_name
        / "model_with_mappings.pt"
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    checkpoint_type = checkpoint.get("model_type")
    if checkpoint_type is not None and checkpoint_type != model_name:
        raise ValueError(
            f"{checkpoint_path} contains {checkpoint_type}, not {model_name}"
        )

    hyperparameters = checkpoint.get("hyperparameters", IMPLICIT_DEFAULTS)
    model = create_implicit_model(
        model_name,
        num_users=len(checkpoint["user_to_index"]),
        num_items=len(checkpoint["item_to_index"]),
        hyperparameters=hyperparameters,
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint["user_to_index"], checkpoint["item_to_index"]


def load_explicit_model():
    checkpoint_path = (
        ARTIFACT_ROOT / "explicit" / "mlp" / "model_with_mappings.pt"
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    hyperparameters = checkpoint.get("hyperparameters", EXPLICIT_DEFAULTS)
    model = ExplicitRecommenderMLP(
        num_users=len(checkpoint["user_to_index"]),
        num_items=len(checkpoint["item_to_index"]),
        embedding_dim=hyperparameters["embedding_dim"],
        hidden_dims=tuple(hyperparameters["hidden_dims"]),
        dropout=hyperparameters["dropout"],
    ).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model, checkpoint["user_to_index"], checkpoint["item_to_index"]


def score_catalog(
    model: torch.nn.Module,
    user_index: int,
    item_to_index: dict,
    score_name: str,
) -> pd.DataFrame:
    index_to_item = [None] * len(item_to_index)
    for item_id, item_index in item_to_index.items():
        index_to_item[item_index] = str(item_id)

    score_batches = []
    with torch.inference_mode():
        for start in range(0, len(index_to_item), INFERENCE_BATCH_SIZE):
            stop = min(start + INFERENCE_BATCH_SIZE, len(index_to_item))
            item_indices = torch.arange(start, stop, device=DEVICE)
            user_indices = torch.full_like(item_indices, user_index)
            scores = model(user_indices, item_indices)
            score_batches.append(scores.cpu())

    return pd.DataFrame(
        {
            ITEM_COLUMN: index_to_item,
            score_name: torch.cat(score_batches).numpy(),
        }
    )


def add_metadata(ranking: pd.DataFrame) -> pd.DataFrame:
    result = ranking.merge(book_metadata, on=ITEM_COLUMN, how="left")
    result.insert(0, "rank", np.arange(1, len(result) + 1))
    return result


def show_ranking(title: str, ranking: pd.DataFrame, score_columns: list[str]):
    print(title)
    columns = [
        "rank",
        "Book-Title",
        "Book-Author",
        "Year-Of-Publication",
        "Publisher",
        ITEM_COLUMN,
        *score_columns,
        "Image-URL-M",
    ]
    display(
        ranking[columns]
        .style
        .format({column: "{:.4f}" for column in score_columns})
        .hide(axis="index")
    )

In [ ]:
def recommend_for_user(
    user_id: int,
    implicit_model_name: str,
    top_k: int = 5,
    explicit_pool_size: int = 1_000,
):
    implicit_model, implicit_users, implicit_items = load_implicit_model(
        implicit_model_name
    )
    explicit_model, explicit_users, explicit_items = load_explicit_model()

    missing_from = []
    if user_id not in implicit_users:
        missing_from.append(f"implicit {implicit_model_name}")
    if user_id not in explicit_users:
        missing_from.append("explicit mlp")
    if missing_from:
        raise ValueError(
            f"User-ID {user_id} is unknown to: {', '.join(missing_from)}. "
            "Choose a User-ID from user_activity."
        )

    seen_isbns = set(
        ratings.loc[ratings[USER_COLUMN] == user_id, ITEM_COLUMN]
    )

    implicit_scores = score_catalog(
        implicit_model,
        implicit_users[user_id],
        implicit_items,
        "implicit_logit",
    )
    implicit_logits = implicit_scores["implicit_logit"].to_numpy()
    implicit_scores["implicit_probability"] = np.exp(
        -np.logaddexp(0.0, -implicit_logits)
    )
    implicit_scores = implicit_scores[
        implicit_scores[ITEM_COLUMN].isin(metadata_isbns)
        & ~implicit_scores[ITEM_COLUMN].isin(seen_isbns)
    ].reset_index(drop=True)

    explicit_scores = score_catalog(
        explicit_model,
        explicit_users[user_id],
        explicit_items,
        "explicit_score",
    )
    explicit_scores = explicit_scores[
        explicit_scores[ITEM_COLUMN].isin(metadata_isbns)
        & ~explicit_scores[ITEM_COLUMN].isin(seen_isbns)
    ].reset_index(drop=True)

    implicit_top = add_metadata(
        implicit_scores.nlargest(top_k, "implicit_probability")
        .drop(columns="implicit_logit")
        .reset_index(drop=True)
    )
    explicit_top = add_metadata(
        explicit_scores.nlargest(top_k, "explicit_score")
        .reset_index(drop=True)
    )

    common_scores = explicit_scores.merge(
        implicit_scores[[ITEM_COLUMN, "implicit_probability"]],
        on=ITEM_COLUMN,
        how="inner",
    )
    explicit_pool = common_scores.nlargest(
        explicit_pool_size,
        "explicit_score",
    ).copy()
    explicit_pool["combined_score"] = (
        explicit_pool["explicit_score"]
        * explicit_pool["implicit_probability"]
    )
    combined_top = add_metadata(
        explicit_pool.nlargest(top_k, "combined_score")
        .reset_index(drop=True)
    )

    print(
        f"User-ID {user_id} | implicit model: {implicit_model_name} | "
        f"excluded seen books: {len(seen_isbns):,}"
    )
    show_ranking(
        "Top 5 — implicit",
        implicit_top,
        ["implicit_probability"],
    )
    show_ranking(
        "Top 5 — explicit",
        explicit_top,
        ["explicit_score"],
    )
    show_ranking(
        f"Top 5 — combined from explicit top {len(explicit_pool):,}",
        combined_top,
        ["explicit_score", "implicit_probability", "combined_score"],
    )

    return {
        "implicit": implicit_top,
        "explicit": explicit_top,
        "combined": combined_top,
    }

In [ ]:
recommendations = recommend_for_user(
    USER_ID,
    IMPLICIT_MODEL_NAME,
    top_k=TOP_K,
    explicit_pool_size=EXPLICIT_POOL_SIZE,
)